# 03 特征筛选模块 (core.selectors)

覆盖所有筛选器的使用方式和 CompositeFeatureSelector 流水线。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import hscredit as hsc
from hscredit import germancredit, init_setting
from hscredit import (
    VarianceSelector, NullSelector, ModeSelector, CorrSelector,
    VIFSelector, IVSelector, LiftSelector, PSISelector,
    CardinalitySelector, TypeSelector, RegexSelector,
    FeatureImportanceSelector, Chi2Selector, FTestSelector,
    MutualInfoSelector, CompositeFeatureSelector,
    SelectionReportCollector,
)
from sklearn.model_selection import train_test_split
init_setting()
df = germancredit()
target = 'creditability'
X = df.drop(columns=[target]); y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
print('特征数:', X_tr.shape[1])

## 1. 基础过滤筛选器

In [ ]:
# 缺失率筛选

In [ ]:
null_sel = NullSelector(threshold=0.5)
null_sel.fit(X_tr, y_tr)
print('NullSelector 保留:', null_sel.get_support(indices=True))

In [ ]:
# 方差筛选
var_sel = VarianceSelector(threshold=0.01)
var_sel.fit(X_tr, y_tr)
print('VarianceSelector 保留:', var_sel.transform(X_tr).columns.tolist())

In [ ]:
# 众数占比筛选
mode_sel = ModeSelector(threshold=0.95)
mode_sel.fit(X_tr, y_tr)
print('ModeSelector 保留特征数:', mode_sel.transform(X_tr).shape[1])

## 2. 相关性与多重共线性

In [ ]:
num_X = X_tr.select_dtypes('number')
corr_sel = CorrSelector(threshold=0.8)
corr_sel.fit(num_X, y_tr)
print('CorrSelector 保留:', corr_sel.transform(num_X).columns.tolist())

In [ ]:
vif_sel = VIFSelector(threshold=10)
vif_sel.fit(num_X, y_tr)
print('VIFSelector 保留:', vif_sel.transform(num_X).columns.tolist())

## 3. 预测能力筛选（IV / Lift / Chi2 / FTest / MI）

In [ ]:
iv_sel = IVSelector(threshold=0.02, max_n_bins=5)
iv_sel.fit(X_tr, y_tr)
print('IV筛选保留:', iv_sel.transform(X_tr).columns.tolist())

In [ ]:
lift_sel = LiftSelector(threshold=1.2, top_pct=0.1)
lift_sel.fit(X_tr, y_tr)
print('Lift筛选保留数:', lift_sel.transform(X_tr).shape[1])

In [ ]:
fi_sel = FeatureImportanceSelector(threshold=0.01, n_estimators=50, random_state=42)
fi_sel.fit(X_tr, y_tr)
print('重要性筛选保留:', fi_sel.transform(X_tr).columns.tolist())

## 4. 稳定性筛选（PSI）

In [ ]:
psi_sel = PSISelector(threshold=0.25, max_n_bins=5)
psi_sel.fit(X_tr, y_tr, X_test=X_te)
print('PSI筛选保留:', psi_sel.transform(X_tr).columns.tolist())

## 5. 类型/正则筛选

In [ ]:
type_sel = TypeSelector(dtype='number')
type_sel.fit(X_tr)
print('数值型特征:', type_sel.transform(X_tr).columns.tolist()[:5])

In [ ]:
regex_sel = RegexSelector(pattern='credit')
regex_sel.fit(X_tr)
print('包含credit的特征:', regex_sel.transform(X_tr).columns.tolist())

## 6. CompositeFeatureSelector — 完整筛选流水线

In [ ]:
collector = SelectionReportCollector()
comp_sel = CompositeFeatureSelector([
    ('null', NullSelector(threshold=0.5)),
    ('mode', ModeSelector(threshold=0.95)),
    ('iv',   IVSelector(threshold=0.02, max_n_bins=5)),
    ('corr', CorrSelector(threshold=0.8)),
], report_collector=collector)
comp_sel.fit(X_tr, y_tr)
X_selected = comp_sel.transform(X_tr)
print('原始特征数:', X_tr.shape[1])
print('筛选后特征数:', X_selected.shape[1])
print('筛选报告:')
collector.report()